In [ ]:
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense


# --------------------------------------------------
# 1. Sample text data
# --------------------------------------------------
text = "This is a sample text for language modeling using RNN."


# --------------------------------------------------
# 2. Create a set of unique characters
# --------------------------------------------------
chars = sorted(set(text))

char_to_index = {
    char: index for index, char in enumerate(chars)
}

index_to_char = {
    index: char for index, char in enumerate(chars)
}

print("Unique characters:", chars)
print("Number of characters:", len(chars))


# --------------------------------------------------
# 3. Convert text into character indices
# --------------------------------------------------
text_indices = [
    char_to_index[char]
    for char in text
]


# --------------------------------------------------
# 4. Create input-output sequences
# --------------------------------------------------
seq_length = 20

sequences = []
next_char = []

for i in range(0, len(text_indices) - seq_length):

    sequences.append(
        text_indices[i:i + seq_length]
    )

    next_char.append(
        text_indices[i + seq_length]
    )


# --------------------------------------------------
# 5. Convert sequences to NumPy arrays
# --------------------------------------------------
X = np.array(sequences)
y = np.array(next_char)

print("X shape:", X.shape)
print("y shape:", y.shape)


# --------------------------------------------------
# 6. Build the RNN model
# --------------------------------------------------
model = Sequential([

    Embedding(
        input_dim=len(chars),
        output_dim=50,
        input_length=seq_length
    ),

    SimpleRNN(
        100,
        return_sequences=False
    ),

    Dense(
        len(chars),
        activation="softmax"
    )
])


# --------------------------------------------------
# 7. Compile the model
# --------------------------------------------------
model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)


# --------------------------------------------------
# 8. Display model architecture
# --------------------------------------------------
model.summary()


# --------------------------------------------------
# 9. Train the model
# --------------------------------------------------
model.fit(
    X,
    y,
    epochs=50,
    batch_size=16
)


# --------------------------------------------------
# 10. Generate text using the trained model
# --------------------------------------------------
seed_text = "This is a sample"

generated_text = seed_text

num_chars_to_generate = 100


for _ in range(num_chars_to_generate):

    # Convert seed text to indices
    seed_indices = [
        char_to_index[char]
        for char in seed_text
        if char in char_to_index
    ]

    # Keep only the last seq_length characters
    seed_indices = seed_indices[-seq_length:]

    # If sequence is shorter than seq_length,
    # pad it with zeros
    if len(seed_indices) < seq_length:

        seed_indices = (
            [0] * (seq_length - len(seed_indices))
            + seed_indices
        )

    # Convert to NumPy array
    seed_array = np.array(seed_indices).reshape(1, -1)

    # Predict next character
    prediction = model.predict(
        seed_array,
        verbose=0
    )

    # Get character index with highest probability
    next_index = np.argmax(prediction[0])

    # Convert index back to character
    next_char = index_to_char[next_index]

    # Add predicted character
    generated_text += next_char

    # Update seed text
    seed_text = seed_text + next_char

    # Keep only last seq_length characters
    seed_text = seed_text[-seq_length:]


# --------------------------------------------------
# 11. Print generated text
# --------------------------------------------------
print("\nGenerated Text:")
print(generated_text)